## Computer Vision Practice (LeNet Architecture for AMNIST Dataset - Digit Detection)

#### Basic Setup

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
"""
                          CV - Computer Vision
                        -------------------------
CNN - Convolutional Neural Networks
=> Convolution Layer (Necessary Information for CNN):
   Input Image
   Image size (I)
   Number of filers/kernels
   Kernel size (n), here, n means n*n size matrix
   stride (s), here, s means, s*s size matrix. Stride is a step of kernel
   padding (p)
=> Action Function, default = tanh
=> Formula: output/Feature map = [(I - n + 2p) / s] + 1
=> Pooling Layer (extract the important features): 
   Kernel size (n)
   stride (n)
=> Apply Batch Normalization
"""

In [14]:
# select device for keeping data and model in same device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [15]:
# Hyperparameters
batch_size = 64
learning_rate = 0.001
num_epochs = 10

In [16]:
# Preprocessing
transform = transforms.Compose([
    transforms.Pad(2), # original size 28 * 28 * 1, Padding 2 in l, r, u, d. So, new size = (2 + 28 + 2) * (2 + 28 + 2) * 1 = 32 * 32 * 1
                       # here, 1 Channel no. for Grayscal image. For, rgb, channel no. 3. i.e. (24 * 24 * 3)
                       # As we'll use LeNet-5 pretrained model, 32 * 32 size image is best for this model based on it's Architecture.
    transforms.ToTensor(), # scale down pixels values from 0 - 255 to 0 - 1
    transforms.Normalize((0.1307,), (0.3081,)) # mean, std normalization, value given by AMNIST Dataset
])

#### Import and Load Dataset [Practice AMNIST dataset given by torch]

In [17]:
# import torch practice datasets
from torchvision import datasets

train_dataset = datasets.MNIST(
    root='../../../../my-practice/Dataset/amnist_data',
    train=True,
    download=True,
    transform=transform # data preprocessing of a single batch done here.
)

test_dataset = datasets.MNIST(
    root='../../../../my-practice/Dataset/amnist_data',
    train=False,
    download=True,
    transform=transform
)

In [18]:
# Load Dataset using DataLoader. DataLoader: Helps to load batch dataset. Create batches, Load data batch by batch to Dataset class for 
# preprocessed and then DataLoader provide the preprocessed data [a single batch] to our model for calculation/prediction.
from torch.utils.data import DataLoader
train_loader = DataLoader(
    dataset=train_dataset, # provide single batch [data] to the dataset for preprocessing. Later, provide the preprocessed batch to our model
    batch_size=64, # create batches with 64 data [size]
    shuffle=True # When I want to distribute randomly
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=False
)

#### LeNet Architecture

In [19]:
# Details in Notes or Web serach for LeNet Architecture
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        # convolution layer 1: 6 filters/Kernel, kernel/filter size(n) 5x5, stride=1(s), padding=0(p)
        self.conv1 = nn.Conv2d( # nn.Conv2d -> for image dataset, nn.Conv3d -> For video dataset
            in_channels=1, # in_channels = 1 -> grayscale image dataset, in_channels = 1 -> rgb image dataset
            out_channels=6, # output channels / Feature map size will be same as no. of filters. 6 filter extracts 6 important features
            kernel_size=5, # (n) odd number square matrix
            stride=1, # (s) step l, r, u, d for calculate feature map
            padding=0 # (p) It will not impact our resutl
        )
        # convolution layer 2: 16 filters, kernel 5x5, stride=1, padding=0
        self.conv2 = nn.Conv2d(
            in_channels=6, # from conv1, out_channels=6 will be input for conv2
            out_channels=16, # same as no. of filters
            kernel_size=5, # (n) filter/kernel size
            stride=1, # (s) steps l, r, u, d
            padding=0  # (p)
        )
        
        # pooling layer 1: kernel size 2x2,  stride=(2,2) -> If we need to reduce image size by removing unnecessary features.
        self.maxpool = nn.MaxPool2d( # MaxPool2d calculate the max value for each image patch [each ] and final result is reduce the main image. 
            kernel_size=2, # using kernel_size=2, stride=2, input image's feature will be reduce by 50% [step 2 u, d, l, r] -> reduce less important feture.
            stride=2,
        )
        
        # ANN/DFF ML Part starts here small no. of fetures value. 
        # fully connected layer 1: takes 5x5x16 -> 120
        self.fc1 = nn.Linear(in_features=5 * 5 * 16, out_features=120) # 120 individual neurons will get (5 * 5 * 16) input and each neuron will generate output (total 120)
        # fully connected layer 2: takes 120 inputs, outputs 84 values
        self.fc2 = nn.Linear(in_features=120, out_features=84) # neurons 84, input 120. so, output = no. of neurons = 84
        # fully connected layer 3: takes 84 inputs, outputs 10 values
        self.fc3 = nn.Linear(in_features=84, out_features=10) # neurons 10, input 84. so, output = no. of neurons = 10 [no. of Digits 0 - 9]
        
        # activation function
        self.relu = nn.ReLU() # take only max(0, x), less than zero [negative value] is unnecessary.

    # Calculation starts here using (Covolution Layer, Pooling layer, DFF/ANN Layer, Activation Function) and return result/prediction  
    def forward(self, x): # x is the input image which is preprocessed and we'll apply CNN, DFF and all other layers on x
        # CNN Part
        # input shape: 32 x 32 x 1
        x = self.conv1(x) # 28 x 28 x 6 -> feture map size =  [(I - n + 2p) / s] + 1 = [(32 - 5 + 2*0) / ] + 1 = 27 + 1 = 28, 
                          # 6-> no. of feature map/output which is same as the no. of Filter/Kernel. Each Filter extracts one Feature map/output. 
        x = self.maxpool(x) # 14 x 14 x 6 -> reduce 50% because of 2*2 size kernel and 2 size stride [steps] but won't change the no. of Feature map/output.
        x = self.relu(x) # 14 x 14 x 6 -> apply activation function [remove unnecessary result/features]
        
        x = self.conv2(x) # 10 x 10 x 16 -> I = 14, n = 5, s = 2, p = 0, Total filter 16. So, [(14 - 5 + 2 * 0) / 1 ] + 1 = 9 + 1 = 10.
        x = self.maxpool(x) # 5 x 5 x 16 -> reduce 50%
        x = self.relu(x) # 5 x 5 x 16 -> activation function
        
        # DFF/ANN Part
        x = x.view(-1, 5 * 5 * 16) # flattens the shape into a vector from  2D matrix[image]. As, we need to apply/feed DFF/ANN. So, we need vector for calculation. 
        
        x = self.fc1(x) # 5 * 5 * 16 input and 120 output
        x = self.relu(x)

        x = self.fc2(x) # 120 input(x) and 84 output
        x = self.relu(x) # 84
        
        x = self.fc3(x) # 84 input(x) and 10 output(no. of digits 0 - 9, final result/prediction)

        return x

In [20]:
# Define model in our device
model = LeNet5().to(device)

# Define Optimizer and Loss/Cost calculator
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=0.001) # used for optimize the loss/cost
criterion = nn.CrossEntropyLoss() # Used for non-binary classification

##### Model Training and Evaluation

In [21]:
for epoch in range(num_epochs):
    model.train() # train mode
    running_loss = 0.0 # loss calculate
    for images, labels in train_loader: # load images and labels [0 - 9 digits] from train loader
        images, labels = images.to(device), labels.to(device) # keep in same device
        outputs = model(images) # result/prediciton
        optimizer.zero_grad() # finding local minimum. [optimal/minimum loss points]
        loss = criterion(outputs, labels) # calculate loss
        loss.backward() # Compute Gradient Descent(Back Propagation)
        optimizer.step() # # update weights(w) and bias(b) using optimal points
        running_loss += loss.item() # sum of loss/cost
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(train_loader):.4f}")

Epoch [1/10], Loss: 0.2316
Epoch [2/10], Loss: 0.0675
Epoch [3/10], Loss: 0.0479
Epoch [4/10], Loss: 0.0375
Epoch [5/10], Loss: 0.0302
Epoch [6/10], Loss: 0.0264
Epoch [7/10], Loss: 0.0231
Epoch [8/10], Loss: 0.0189
Epoch [9/10], Loss: 0.0169
Epoch [10/10], Loss: 0.0156


In [ ]:
model.eval() # evaluation mode
correct = 0
total = 0
with torch.no_grad(): # no need gradient calculation
    for images, labels in test_loader: # load from test loader
        images, labels = images.to(device), labels.to(device) # same device
        outputs = model(images) # prediction
        _, predicted = torch.max(outputs.data, 1) # each neuron has final 10 outputs probability of 0 - 9. max value is the most probability of correct predition.
        total += labels.size(0) # total prediction
        correct += (predicted == labels).sum().item() # correct preditction
print(f"Accuracy on the test set: {100 * correct / total:.2f}%")

Accuracy on the test set: 99.03%
